In [43]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))


# Conversion and loading UVV files

In [44]:
import glob
import os
import pandas as pd
import yaml
import copy
# Install ruamel.yaml if not already installed
#%pip install ruamel.yaml
from ruamel.yaml import YAML
import pprint

Select folder from where you would like to convert the files.

In [45]:
input_folder = 'data/pre_metadata_processed/'
csv_files = glob.glob(input_folder + '*.csv')
len(csv_files)
input_folder_yaml = 'data/pre_metadata_processed/'
yaml_files = glob.glob(input_folder_yaml + '*.yaml')
len(yaml_files)
if len(csv_files) == len(yaml_files):
    print ("Die Länge der csv-files entspricht der der yaml-files")

Die Länge der csv-files entspricht der der yaml-files


In [46]:
import os
from ruamel.yaml import YAML
import shutil

# Pair csv and yaml files by their base filename (without extension)
csv_basenames = {os.path.splitext(os.path.basename(f))[0]: f for f in csv_files}
yaml_basenames = {os.path.splitext(os.path.basename(f))[0]: f for f in yaml_files}
outdir = 'data/pre_metadata_processed/sorted/'
paired_files = []
for base in csv_basenames:
    if base in yaml_basenames:
        paired_files.append({'base': base, 'csv': csv_basenames[base], 'yaml': yaml_basenames[base]})

# Read irradiation.time.value from each yaml file
yaml_ruamel = YAML()
for pair in paired_files:
    with open(pair['yaml'], 'r', encoding='utf-8') as f:
        data = yaml_ruamel.load(f)
        irr_time = None
        try:
            irr_time = float(data['irradiation']['time']['value'])
        except Exception:
            irr_time = data['irradiation']['time']['value']
        pair['irradiation_time'] = irr_time

# Sort by irradiation_time
# Sort by irradiation_time, converting "-" to "." if present
def get_sort_key(x):
    irr = x['irradiation_time']
    # If it's a string and contains "-", replace with "."
    if isinstance(irr, str) and "-" in irr:
        try:
            return float(irr.replace("-", "."))
        except Exception:
            return irr
    return float(irr)

paired_files_sorted = sorted(paired_files, key=get_sort_key)



paired_files_sorted

[{'base': '20240201_UVV_ALLA_LN46o0001_1',
  'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_1.csv',
  'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_1.yaml',
  'irradiation_time': 0.0},
 {'base': '20240201_UVV_ALLA_LN46o0002_1',
  'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0002_1.csv',
  'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0002_1.yaml',
  'irradiation_time': 0.0},
 {'base': '20240201_UVV_ALLA_LN46o0003_1',
  'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0003_1.csv',
  'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0003_1.yaml',
  'irradiation_time': 0.0},
 {'base': '20240201_UVV_ALLA_LN46o0004_1',
  'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0004_1.csv',
  'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0004_1.yaml',
  'irradiation_time': 0.0},
 {'base': '20240208_UVV_ALLA_LN46o0005_1',
  'csv': '../../data/202402_procc

In [47]:
from collections import defaultdict

# Helper to extract batchname from base
def extract_batchname(base):
    # Example: '20240214_UVV_ALLA_LN31o0001_7' -> 'LN31o0001'
    return base.split('_')[-2]

# Group paired files by batchname
batch_dict = defaultdict(list)
for pair in paired_files_sorted:
    batchname = extract_batchname(pair['base'])
    batch_dict[batchname].append(pair)

# Show all entries for batchname 'LN46o0001'
for batchname, entries in batch_dict.items():
    print(f"Batch: {batchname}")
    sorted_entries = sorted(entries, key=lambda x: float(str(x['irradiation_time']).replace("-", ".")))
    outdir_new = 'data/pre_metadata_processed/sorted/'
    os.makedirs(outdir_new, exist_ok=True)
    for new_idx, entry in enumerate(sorted_entries, 1):
        base_no_num = '_'.join(entry['base'].split('_')[:-1])
        new_base = f"{base_no_num}_{new_idx}"
        csv_new_path = os.path.join(outdir_new, os.path.basename(new_base + '.csv'))
        yaml_new_path = os.path.join(outdir_new, os.path.basename(new_base + '.yaml'))
        shutil.copy2(entry['csv'], csv_new_path)
        shutil.copy2(entry['yaml'], yaml_new_path)
        print(entry)


Batch: LN46o0001
{'base': '20240201_UVV_ALLA_LN46o0001_1', 'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_1.csv', 'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_1.yaml', 'irradiation_time': 0.0}
{'base': '20240201_UVV_ALLA_LN46o0001_2', 'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_2.csv', 'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_2.yaml', 'irradiation_time': 1.0}
{'base': '20240201_UVV_ALLA_LN46o0001_4', 'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_4.csv', 'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_4.yaml', 'irradiation_time': 2.0}
{'base': '20240201_UVV_ALLA_LN46o0001_5', 'csv': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_5.csv', 'yaml': '../../data/202402_proccesed_csv\\20240201_UVV_ALLA_LN46o0001_5.yaml', 'irradiation_time': 3.0}
{'base': '20240201_UVV_ALLA_LN46o0001_6', 'csv': '../../data/202402_proccesed_csv\\20240201

In [48]:
for batchname, entries in batch_dict.items():
    sorted_entries = sorted(entries, key=lambda x: float(str(x['irradiation_time']).replace("-", ".")))
    for idx, entry in enumerate(sorted_entries, 1):
        base_no_num = '_'.join(entry['base'].split('_')[:-1])
        new_base = f"{base_no_num}_{idx}"
        csv_new_path = os.path.join(outdir, os.path.basename(new_base + '.csv'))
        yaml_new_path = os.path.join(outdir, os.path.basename(new_base + '.yaml'))
        shutil.copy2(entry['csv'], csv_new_path)
        shutil.copy2(entry['yaml'], yaml_new_path)
        entry['csv_new'] = csv_new_path
        entry['yaml_new'] = yaml_new_path
